# Week 8, notebook 2: Unconfined flow and dry cells

**HWRS 564a · Fall 2026**

Every model so far has been confined. Most of the Tucson basin is a water-table
aquifer, so this notebook switches the layer to unconfined and looks at what
changes: the solver needs more care, and cells can go dry.

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked `# YOUR TURN` have something
for you to write. Cells marked `# CHECK` test your answer and raise an error if
it is wrong.

## Learning objectives

By the end of this notebook you can:

1. Say what `laytyp=0` and `laytyp=1` mean and when each one applies
2. Find out from the `.list` file why a run did not converge
3. Choose solver tolerances that suit the size of the flows in the model
4. Find dry cells in the output and list what to check when you get them

---

## Part 1: Setup

In [ ]:
from pathlib import Path

import flopy
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("../..")  # course repo root
MF_EXE = ROOT / "modflow" / "mf2005"
assert MF_EXE.exists(), (
    f"MODFLOW binary not found at {MF_EXE}. Run ./postbuild.sh from a terminal."
)

RUN = ROOT / "_run"
NROW, NCOL, DELR = 40, 60, 250.0
WELL_ROW, WELL_COL = 20, 30
EXTENT = [0, NCOL * DELR / 1000, 0, NROW * DELR / 1000]   # km, for plotting

print("ready")

---

## Part 2: Confined and unconfined layers

The LPF package has an argument `laytyp` that sets the layer type.

**`laytyp=0`, confined.** The aquifer is full from top to bottom. Its
transmissivity is

$$T = K b$$

where $b$ is the layer thickness. $T$ does not depend on head.

**`laytyp=1`, convertible.** If the head is below the top of the layer, the
layer is treated as unconfined. Only the part below the water table carries
water, so

$$T = K (h - z_{bot})$$

Now $T$ depends on the head $h$, and $h$ is what MODFLOW is solving for. The
solver handles this by repeating: guess the heads, calculate $T$, solve for new
heads, calculate $T$ again, and so on until the heads stop changing.

Use `laytyp=1` when the top of the saturated zone is a water table. That is the
case for most of the Tucson basin.

To compare the two we use the rectangular model from Tuesday at 250 m, with two
changes: the layer top is at 700 m so the head stays below it, and the east
boundary is at 680 m. The function takes `laytyp` as an argument, so the
confined and unconfined runs are identical in every other way.

In [ ]:
def run_box(name, laytyp, well_q=-20000.0, botm=400.0, west_head=620.0,
            start_head=660.0, pcg_kwargs=None):
    """Build and run the rectangular model. Returns (success, head, workspace)."""
    ws = RUN / f"week08_{name}"
    ws.mkdir(parents=True, exist_ok=True)

    mf = flopy.modflow.Modflow("box", model_ws=str(ws), exe_name=str(MF_EXE))
    flopy.modflow.ModflowDis(mf, 1, NROW, NCOL, delr=DELR, delc=DELR,
                             top=700.0, botm=botm, nper=1, steady=True)

    ibound = np.ones((1, NROW, NCOL), dtype=int)
    ibound[:, :, 0] = -1
    ibound[:, :, -1] = -1
    strt = np.full((1, NROW, NCOL), start_head)
    strt[:, :, 0] = west_head
    strt[:, :, -1] = 680.0
    flopy.modflow.ModflowBas(mf, ibound=ibound, strt=strt)

    flopy.modflow.ModflowLpf(mf, hk=12.0, laytyp=laytyp, ipakcb=53)
    flopy.modflow.ModflowWel(
        mf, stress_period_data={0: [[0, WELL_ROW, WELL_COL, well_q]]}, ipakcb=53)
    flopy.modflow.ModflowRch(mf, rech=1.2e-4, ipakcb=53)
    flopy.modflow.ModflowPcg(mf, **(pcg_kwargs or {}))
    flopy.modflow.ModflowOc(mf, stress_period_data={(0, 0): ["save head", "save budget"]})

    mf.write_input()
    ok, buff = mf.run_model(silent=True, report=True)
    head = flopy.utils.HeadFile(str(ws / "box.hds")).get_data()
    return ok, head, ws


ok_conf, head_conf, ws_conf = run_box("confined", laytyp=0)
print(f"confined:   success = {ok_conf}")

ok_uncf, head_uncf, ws_uncf = run_box("unconfined", laytyp=1)
print(f"unconfined: success = {ok_uncf}")

The confined model runs. The unconfined model, with the same solver settings,
does not converge.

---

## Part 3: Why it did not converge

The `.list` file has the details. First, the settings the solver used:

In [ ]:
listing = (ws_uncf / "box.list").read_text().splitlines()

for line in listing:
    if "CALLS TO PCG" in line or "CRITERION FOR CLOSURE" in line or "FAILED" in line:
        print(line.strip())

MODFLOW's PCG solver stops when two conditions are both met:

- the largest head change between iterations is below `hclose` (in metres)
- the largest residual is below `rclose`. The residual is the water-balance
  error for one cell, in m³/d.

FloPy's defaults are `hclose=1e-5` and `rclose=1e-5`. It also limits the solver
to `mxiter=50` outer iterations, each of which recalculates $T$.

Now look at the size of the flows in this model:

In [ ]:
budget = flopy.utils.MfListBudget(str(ws_uncf / "box.list")).get_data()
for row in budget:
    name = row["name"].decode()
    if name in ("CONSTANT_HEAD_IN", "RECHARGE_IN", "WELLS_OUT", "CONSTANT_HEAD_OUT",
                "IN-OUT", "PERCENT_DISCREPANCY"):
        print(f"{name:22s} {row['value']:14.4f}")

About 140,000 m³/d moves through the model, and the budget balances to a
fraction of a cubic metre per day. But `rclose=1e-5` asks for every cell to
balance to 0.00001 m³/d. In the unconfined model the solver gets close to that
and then stops improving, because the numbers it is working with are not that
precise.

### YOUR TURN 1

A common first guess is that the solver needs more iterations. Test it: run
the unconfined model again with `mxiter=200`.

- `ok_more`, `head_more`, `ws_more`: the result of `run_box` with the name
  `"unconfined_more"`, `laytyp=1`, and `pcg_kwargs={"mxiter": 200}`

In [ ]:
# YOUR TURN
ok_more, head_more, ws_more = ...

In [ ]:
# CHECK
assert ok_more is False, f"expected this run to fail as well, got success = {ok_more}"
assert "MAXIMUM NUMBER OF CALLS TO PCG ROUTINE =      200" in (ws_more / "box.list").read_text(), (
    "the .list file does not show mxiter = 200"
)
print(f"mxiter=200: success = {ok_more}")

More iterations do not help, because the residual is not getting any smaller.
The fix is to ask for a tolerance that makes sense for this model. A residual
of 1 m³/d is less than one part in 100,000 of the flow through the model.

In [ ]:
ok_fixed, head_fixed, ws_fixed = run_box(
    "unconfined_fixed", laytyp=1, pcg_kwargs={"rclose": 1.0}
)
print(f"rclose=1.0: success = {ok_fixed}")

### YOUR TURN 2

The first unconfined run "failed" and this one "succeeded". Find out how
different their heads are.

- `max_difference`: the largest absolute difference between `head_uncf` and
  `head_fixed`, over all cells

In [ ]:
# YOUR TURN
max_difference = ...

In [ ]:
# CHECK
assert abs(max_difference - np.abs(head_uncf - head_fixed).max()) < 1e-9, f"got {max_difference}"
assert max_difference < 0.01
print(f"largest head difference: {max_difference:.6f} m")

The two runs give the same heads. The first one had already reached the answer.
It was reported as a failure only because the tolerance could not be met.

That will not always be true. A run that fails to converge can also be far from
any answer, as the first basin model was on Tuesday. So when a run fails:

1. Read the solver section and the budget in the `.list` file.
2. Compare `hclose` and `rclose` with the heads and flows in the model.
3. Change one setting at a time and note what you tried.

The PCG arguments you are most likely to change:

| Argument | FloPy default | Meaning |
|---|---|---|
| `mxiter` | 50 | maximum outer iterations; $T$ is recalculated in each one |
| `iter1` | 30 | maximum inner iterations per outer iteration |
| `hclose` | 1e-5 | head-change tolerance, in the model's length unit |
| `rclose` | 1e-5 | residual tolerance, in the model's flow unit |
| `relax` | 1.0 | relaxation factor; values slightly below 1 can help |

A run that converges has met its tolerances. That says nothing about whether
the boundary conditions, conductivity, or recharge are right.

---

## Part 4: How much difference does `laytyp` make?

In an unconfined aquifer, pumping lowers the water table, which reduces the
saturated thickness, which reduces $T$, which increases the drawdown. The
effect grows with the pumping rate. Here are both layer types with the well
pumping four times as much, 80,000 m³/d.

In [ ]:
ok_c, head_conf_80, _ = run_box("confined_80", laytyp=0, well_q=-80000.0)
ok_u, head_uncf_80, _ = run_box("unconfined_80", laytyp=1, well_q=-80000.0,
                                pcg_kwargs={"rclose": 1.0})
print(f"confined: success = {ok_c}    unconfined: success = {ok_u}")

### YOUR TURN 3

Compare the head in the well cell.

- `confined_well_head`: from `head_conf_80`
- `unconfined_well_head`: from `head_uncf_80`
- `difference_m`: unconfined minus confined

The well is in layer 0, row `WELL_ROW`, column `WELL_COL`.

In [ ]:
# YOUR TURN
confined_well_head = ...
unconfined_well_head = ...
difference_m = ...

In [ ]:
# CHECK
assert abs(confined_well_head - head_conf_80[0, WELL_ROW, WELL_COL]) < 1e-6, f"got {confined_well_head}"
assert abs(unconfined_well_head - head_uncf_80[0, WELL_ROW, WELL_COL]) < 1e-6, f"got {unconfined_well_head}"
assert abs(difference_m - (unconfined_well_head - confined_well_head)) < 1e-6
assert difference_m < -1.0, "the unconfined head should be more than a metre lower"
print(f"confined   {confined_well_head:8.2f} m")
print(f"unconfined {unconfined_well_head:8.2f} m")
print(f"difference {difference_m:+8.2f} m")

The unconfined model has the lower head at the well. The confined model
assumes all 300 m of the layer carries water. In the unconfined model only the
saturated part does, which here is about 230 to 280 m, and it gets thinner
toward the well.

Which one to use is a question about the aquifer. If a confining unit keeps the
aquifer full, use `laytyp=0`. If the top of the saturated zone is a water
table, use `laytyp=1`.

---

## Part 5: Dry cells

If the head in an unconfined cell falls below the bottom of the layer, the cell
has no saturated thickness. MODFLOW marks it dry, gives it a head of `-1e30`,
and takes it out of the calculation.

To see this, make the aquifer much thinner (bottom at 640 m) and pump
40,000 m³/d.

In [ ]:
ok_dry, head_dry, ws_dry = run_box(
    "dry", laytyp=1, well_q=-40000.0, botm=640.0, west_head=660.0,
    start_head=670.0, pcg_kwargs={"mxiter": 200, "rclose": 1.0},
)
print(f"success = {ok_dry}")
print(f"head range in the output: {head_dry.min():.3e} to {head_dry.max():.1f}")

### YOUR TURN 4

`-1e30` is a flag, not a head. Find the dry cells and mask them.

- `dry_mask`: a 2D boolean array, `True` where the cell is dry (head below
  `-1e29`)
- `n_dry`: the number of dry cells
- `head_masked`: a copy of `head_dry[0]` with the dry cells set to `np.nan`

Compare with a threshold instead of testing `== -1e30`. Floating-point values
should not be tested for exact equality.

In [ ]:
# YOUR TURN
dry_mask = ...
n_dry = ...
head_masked = ...

In [ ]:
# CHECK
assert dry_mask.shape == (NROW, NCOL), f"expected a 2D mask, got {dry_mask.shape}"
assert n_dry == (head_dry[0] < -1e29).sum() and n_dry > 0, f"got {n_dry}"
assert np.isnan(head_masked[dry_mask]).all(), "dry cells should be NaN"
assert not np.isnan(head_masked[~dry_mask]).any(), "wet cells should keep their values"
print(f"{n_dry} dry cells ({100 * n_dry / dry_mask.size:.1f}% of the grid)")
print(f"head in the wet cells: {np.nanmin(head_masked):.1f} to {np.nanmax(head_masked):.1f} m")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))

im0 = axes[0].imshow(head_dry[0], cmap="Blues_r", extent=EXTENT)
fig.colorbar(im0, ax=axes[0], label="head (m)")
axes[0].set_title("Head as written by MODFLOW")

im1 = axes[1].imshow(head_masked, cmap="Blues_r", extent=EXTENT)
axes[1].contourf(np.flipud(dry_mask.astype(float)), levels=[0.5, 1.5],
                 colors=["#AB0520"], alpha=0.8, extent=EXTENT)
fig.colorbar(im1, ax=axes[1], label="head (m)")
axes[1].set_title("Dry cells masked (shown in red)")

for ax in axes:
    ax.set_xlabel("distance east (km)")
    ax.set_ylabel("distance north (km)")
plt.tight_layout()
plt.show()

In the left panel the colour scale runs from `-1e30` to 680, so every wet cell
is the same colour. Mask flag values before plotting or calculating anything.

Three values to keep apart:

| Value in the head file | Meaning |
|---|---|
| `hnoflo` (`-999.99` by default) | inactive cell, `ibound = 0` |
| `hdry` (`-1e30` by default) | cell went dry during the run |
| anything else | a calculated head |

Masking makes the plot readable, but the model still has a problem. The well
is in a dry cell, and MODFLOW stops pumping from a cell once it is dry. A dry
cell also carries no water, so its neighbours lose a source and may go dry too.
MODFLOW-2005 does not rewet a dry cell unless you turn on the rewetting
option, which often causes convergence problems of its own.

If a model has dry cells, check these:

- Is the pumping rate realistic for that location?
- Is the layer bottom too shallow there?
- Should the aquifer be split into more layers, so the water table can drop
  through the top one?
- Would MODFLOW-NWT be a better choice? It was written for unconfined problems
  where cells dry and rewet.

In [ ]:
budget_dry = flopy.utils.MfListBudget(str(ws_dry / "box.list")).get_data()
wells_out = [row["value"] for row in budget_dry if row["name"].decode() == "WELLS_OUT"][0]
print(f"pumping asked for: 40,000 m3/d")
print(f"pumping in the budget: {abs(wells_out):,.0f} m3/d")

The budget shows the pumping that MODFLOW applied. Compare it with what you
asked for whenever a model has dry cells.

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks
3. Save

## Next: the Pinal County model

Project 3 uses a groundwater model of Pinal County. The third notebook this
week, `week08_03_pinal_startup.ipynb`, gets the model onto your machine and
loads it, so you can look at its grid, its `ibound`, and its layer types.
Everything in it is something you did this week on the small model.

## Next week

Models that change through time: stress periods, storage, and wells that
switch on and off.

## Stuck?

- This notebook runs seven MODFLOW models. Each takes a second or two.
- `success = False` from the first unconfined run and from YOUR TURN 1 is
  expected.
- If a run fails after you change `rclose`, read the last lines of the `.list`
  file in its folder under `_run/`.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 322B.